# Latent Space Dynamics of Swarm Agents
This notebook simulates swarm behavior using Boids, compresses it with a Graph Autoencoder,
and forecasts latent space dynamics with an LSTM.

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import MinMaxScaler

/home/machi/Projects/Latent-Space-Inference-of-Dynamic-Multi-Agent-Systems/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# SECTION 1: Boids Simulation


num_boids = 20
timesteps = 200
width, height = 640, 480
max_speed = 4
neighborhood_radius = 75

positions = np.random.rand(num_boids, 2) * [width, height]
velocities = (np.random.rand(num_boids, 2) - 0.5) * max_speed
history = []

def limit_speed(v, max_speed):
    speed = np.linalg.norm(v)
    return (v / speed) * max_speed if speed > max_speed else v

def update_boids(pos, vel):
    new_pos, new_vel = pos.copy(), vel.copy()
    for i in range(len(pos)):
        neighbors = [j for j in range(len(pos)) if i != j and np.linalg.norm(pos[i] - pos[j]) < neighborhood_radius]
        if neighbors:
            center = np.mean(pos[neighbors], axis=0)
            separation = np.sum(pos[i] - pos[neighbors], axis=0)
            avg_vel = np.mean(vel[neighbors], axis=0)
            new_vel[i] += (center - pos[i]) * 0.01 + separation * 0.05 + (avg_vel - vel[i]) * 0.05
        new_vel[i] = limit_speed(new_vel[i], max_speed)
        new_pos[i] = np.mod(new_pos[i] + new_vel[i], [width, height])
    return new_pos, new_vel

for t in range(timesteps):
    state = np.hstack((positions, velocities))
    history.append(state)
    positions, velocities = update_boids(positions, velocities)

history = np.array(history)

In [5]:
# SECTION 2: Graph Construction
def get_edge_index(positions, threshold=75):
    edges = []
    for i in range(len(positions)):
        for j in range(i+1, len(positions)):
            if np.linalg.norm(positions[i] - positions[j]) < threshold:
                edges.append((i, j))
                edges.append((j, i))
    if not edges:  # Return an empty tensor with shape (2, 0) if no edges
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).t()

scaler = MinMaxScaler()
history_scaled = history.reshape(-1, 4)
history_scaled = scaler.fit_transform(history_scaled)
history_scaled = history_scaled.reshape(timesteps, num_boids, 4)

graph_data = []
for t in range(timesteps):
    x = torch.tensor(history_scaled[t], dtype=torch.float)
    edge_index = get_edge_index(history[t, :, :2])
    data = Data(x=x, edge_index=edge_index)
    graph_data.append(data)

In [6]:
# SECTION 3: Graph Autoencoder
class GraphEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=8):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 16)
        self.conv2 = GCNConv(16, out_channels)
    def forward(self, x, edge_index):
        # Check if edge_index is empty
        if edge_index.numel() == 0:
            # Handle the case of an empty graph, e.g., return a tensor of zeros
            # with the same shape as the expected output but with the latent dimension
            batch_size = x.size(0)
            return torch.zeros(batch_size, self.conv2.out_channels, device=x.device)
        x = torch.relu(self.conv1(x, edge_index))
        # Check again before the second convolution
        if edge_index.numel() == 0:
             batch_size = x.size(0)
             return torch.zeros(batch_size, self.conv2.out_channels, device=x.device)
        return self.conv2(x, edge_index)


class GraphDecoder(nn.Module):
    def __init__(self, in_channels=8, out_channels=4):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, 16)
        self.lin3 = nn.Linear(16, 16)
        self.lin4 = nn.Linear(16, 16)
        self.lin2 = nn.Linear(16, out_channels)
    def forward(self, z):
        return self.lin2(torch.relu(self.lin1(z)))

encoder = GraphEncoder()
decoder = GraphDecoder()
ae_optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(50):
    total_loss = 0
    for data in graph_data:
        ae_optimizer.zero_grad()
        z = encoder(data.x, data.edge_index)
        x_hat = decoder(z)
        loss = loss_fn(x_hat, data.x)
        loss.backward()
        ae_optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 13.1586
Epoch 2, Loss: 6.8157
Epoch 3, Loss: 6.7380
Epoch 4, Loss: 6.7442
Epoch 5, Loss: 6.7420
Epoch 6, Loss: 6.7506
Epoch 7, Loss: 6.7561
Epoch 8, Loss: 6.7527
Epoch 9, Loss: 6.7541
Epoch 10, Loss: 6.7601
Epoch 11, Loss: 6.7673
Epoch 12, Loss: 6.7784
Epoch 13, Loss: 6.7870
Epoch 14, Loss: 6.8263
Epoch 15, Loss: 6.8192
Epoch 16, Loss: 6.7811
Epoch 17, Loss: 6.7636
Epoch 18, Loss: 6.7689
Epoch 19, Loss: 6.7719
Epoch 20, Loss: 6.7638
Epoch 21, Loss: 6.7551
Epoch 22, Loss: 6.7646
Epoch 23, Loss: 6.7658
Epoch 24, Loss: 6.7799
Epoch 25, Loss: 6.7747
Epoch 26, Loss: 6.7613
Epoch 27, Loss: 6.7476
Epoch 28, Loss: 6.7494
Epoch 29, Loss: 6.7485
Epoch 30, Loss: 6.7502
Epoch 31, Loss: 6.7522
Epoch 32, Loss: 6.7535
Epoch 33, Loss: 6.7594
Epoch 34, Loss: 6.7583
Epoch 35, Loss: 6.7424
Epoch 36, Loss: 6.7555
Epoch 37, Loss: 6.7425
Epoch 38, Loss: 6.7429
Epoch 39, Loss: 6.7443
Epoch 40, Loss: 6.7425
Epoch 41, Loss: 6.7437
Epoch 42, Loss: 6.7488
Epoch 43, Loss: 6.7754
Epoch 44, Loss: 6.7

In [7]:
# SECTION 4: LSTM in Latent Space
latent_seq = []
with torch.no_grad():
    for data in graph_data:
        z = encoder(data.x, data.edge_index).mean(dim=0)
        latent_seq.append(z)
latent_seq = torch.stack(latent_seq)

seq_len = 10
X_lstm, Y_lstm = [], []
for i in range(len(latent_seq) - seq_len):
    X_lstm.append(latent_seq[i:i+seq_len])
    Y_lstm.append(latent_seq[i+seq_len])
X_lstm, Y_lstm = torch.stack(X_lstm), torch.stack(Y_lstm)

class LatentLSTM(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=64, num_layers=7):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, input_dim)
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

lstm_model = LatentLSTM()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.005)
for epoch in range(40):
    lstm_optimizer.zero_grad()
    output = lstm_model(X_lstm)
    loss = loss_fn(output, Y_lstm)
    loss.backward()
    lstm_optimizer.step()
    print(f"LSTM Epoch {epoch+1}, Loss: {loss.item():.4f}")

LSTM Epoch 1, Loss: 0.0164
LSTM Epoch 2, Loss: 0.0078
LSTM Epoch 3, Loss: 0.0057
LSTM Epoch 4, Loss: 0.0052
LSTM Epoch 5, Loss: 0.0051
LSTM Epoch 6, Loss: 0.0053
LSTM Epoch 7, Loss: 0.0051
LSTM Epoch 8, Loss: 0.0048
LSTM Epoch 9, Loss: 0.0047
LSTM Epoch 10, Loss: 0.0048
LSTM Epoch 11, Loss: 0.0049
LSTM Epoch 12, Loss: 0.0048
LSTM Epoch 13, Loss: 0.0048
LSTM Epoch 14, Loss: 0.0048
LSTM Epoch 15, Loss: 0.0047
LSTM Epoch 16, Loss: 0.0047
LSTM Epoch 17, Loss: 0.0047
LSTM Epoch 18, Loss: 0.0047
LSTM Epoch 19, Loss: 0.0047
LSTM Epoch 20, Loss: 0.0047
LSTM Epoch 21, Loss: 0.0047
LSTM Epoch 22, Loss: 0.0047
LSTM Epoch 23, Loss: 0.0047
LSTM Epoch 24, Loss: 0.0047
LSTM Epoch 25, Loss: 0.0047
LSTM Epoch 26, Loss: 0.0047
LSTM Epoch 27, Loss: 0.0047
LSTM Epoch 28, Loss: 0.0047
LSTM Epoch 29, Loss: 0.0047
LSTM Epoch 30, Loss: 0.0047
LSTM Epoch 31, Loss: 0.0047
LSTM Epoch 32, Loss: 0.0047
LSTM Epoch 33, Loss: 0.0047
LSTM Epoch 34, Loss: 0.0047
LSTM Epoch 35, Loss: 0.0047
LSTM Epoch 36, Loss: 0.0047
L

In [8]:
# SECTION 5: Forecast Evaluation
with torch.no_grad():
    predicted_latents = lstm_model(X_lstm)
    decoded_predictions = decoder(predicted_latents)

gt = graph_data[seq_len].x.mean(dim=0)
pred = decoded_predictions.mean(dim=0)
error = torch.norm(gt - pred).item()
print(f"Reconstruction Error (forecasted): {error:.4f}")

Reconstruction Error (forecasted): 0.0966
